In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

In [ ]:
plt.rcParams['font.sans-serif']=['SimHei']#字体设置

In [ ]:
plt.rcParams['axes.unicode_minus']=False

In [ ]:
df=pd.read_csv(filepath_or_buffer="../output/data_clean.csv",encoding="utf-8") 
#用一个变量来接受读取的信息

In [ ]:
df["date"]=pd.to_datetime(df["date"])#将格式确定为日期，防止为字符串而无法计算

In [ ]:
first_day=df.groupby('user_id')["date"].min().reset_index()#groupby执行后会以user_id设置为索引，reset_index将索引变为普通列

In [ ]:
first_day.columns=['user_id','first_date']

In [ ]:
df=df.merge(first_day,on='user_id')

In [ ]:
df['day_diff']=(pd.to_datetime(df['date'])-pd.to_datetime(df['first_date'])).dt.days#计算距离首日的天数

In [ ]:
retention=df.groupby(['first_date','day_diff'])['user_id'].nunique().reset_index()#分组统计

In [ ]:
retention_pivot=retention.pivot(index='first_date',columns='day_diff',values='user_id')
# 先补齐队列与日龄，再仅对已到观察日的空值补零。
retention_pivot=retention_pivot.reindex(columns=range((df['date'].max()-df['first_date'].min()).days+1))
for cohort in retention_pivot.index:
    for age in retention_pivot.columns:
        observed = pd.Timestamp(cohort)+pd.Timedelta(days=int(age)) <= df['date'].max()
        if observed and pd.isna(retention_pivot.loc[cohort,age]):
            retention_pivot.loc[cohort,age]=0
        elif not observed:
            retention_pivot.loc[cohort,age]=float('nan')


In [ ]:
retention_rate=retention_pivot.div(retention_pivot[0],axis=0)#.div:pandas里面的除法
#`retention_pivot[0]`：第 0 天的用户数，也就是这个同期群**初始总新增人数**，axis=0：按行除法（非常关键！！！）

In [ ]:
retention_rate

In [ ]:
#上面得到的retention_rate是宽表，不能直接给BI
#新增宽表➡长表（melt逆透视，适配FineBI）

In [ ]:
retention_rate_reset=retention_rate.reset_index()

In [ ]:
retention_long=retention_rate_reset.melt(
id_vars='first_date',#标识变量，保持不动的列
var_name='day_diff',
#原来宽表里的列名（0、1、2、3…… 这些天数），全部拆出来，放到新的一列，这一列起名叫 `day_diff`
value_name='retention_rate')#原来宽表里格子里面的留存率数值，全部拆出来放到新列，列名 `retention_rate`

In [ ]:
retention_long['is_observable']=retention_long['first_date']+pd.to_timedelta(retention_long['day_diff'].astype(int),unit='D')<=df['date'].max()
cohort_sizes=df.groupby('first_date')['user_id'].nunique()
retention_long['cohort_users']=retention_long['first_date'].map(cohort_sizes)
retention_long['return_users']=(retention_long['retention_rate']*retention_long['cohort_users']).round()


In [ ]:
retention_long['day_diff']=retention_long['day_diff'].astype(int)#把 day_diff 转成整数

In [ ]:
# 保留尚不可观察的行，用is_observable区分缺失与零留存。
retention_long.to_csv('../output/retention_revised.csv',index=False,encoding='utf-8-sig')
